In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
import subprocess as sp
from scipy import ndimage
from scipy.ndimage.interpolation import zoom
import segmentation_models_pytorch as smp
import pandas as pd
from sklearn.metrics import precision_recall_curve

In [ ]:
def compute_stats(true, preds, cutoff):
    preds_binary = torch.Tensor(preds>cutoff).long()
    tp, fp, fn, tn = smp.metrics.get_stats(
        preds_binary,
        true,
        mode="binary")

    iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
    f1 = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")
    prec = smp.metrics.precision(tp, fp, fn, tn, reduction="micro")
    recall = smp.metrics.recall(tp, fp, fn, tn, reduction="micro")

    return np.array([iou, f1, prec, recall])

def get_objects(ar, pred_thresh=0.5):
    labeled_ar, num_objects = ndimage.label(ar>pred_thresh)
    return labeled_ar, num_objects


def calculate_iou(maska, maskb):
  # Calculates the Intersection over Union (IoU) of two bounding boxes.
  top = np.sum(maska * maskb)
  bottom = np.sum(np.max([maska, maskb], axis=0))
  return top/bottom

def calculate_object_p(truth_masks, pred_masks, pred_thresh=0.5, iou_thresh=0.5):
    true_positives = 0
    false_positives = 0
    total_objects = 0

    for i in range(len(truth_masks)):
        labeled_truth, max_truth = get_objects(truth_masks[i])
        labeled_pred, max_pred = get_objects(pred_masks[i], pred_thresh)
        total_objects += max_truth
        if max_pred > 0:
            labeled_pred_truth_objects = ndimage.labeled_comprehension(
                    labeled_truth,
                    labeled_pred,
                    np.arange(1, max_pred+1),
                    np.unique,
                    np.ndarray,
                    [0])
            for vals_ar in labeled_pred_truth_objects:
                if np.max(vals_ar)==0:
                    false_positives+=1
            for j in range(1, max_truth+1):
                mask_truth = (labeled_truth==j)
                pred_object_vals = [k+1 for k in np.arange(max_pred) if (np.max(labeled_pred_truth_objects[k]==j)>0)]
                if len(pred_object_vals) > 0:
                    mask_pred = np.sum(np.stack([labeled_pred == pov for pov in pred_object_vals], axis=0), axis=0)
                    if np.sum(mask_pred>0) > 4:
                        obj_iou = calculate_iou(mask_truth, mask_pred)
                        if obj_iou > iou_thresh:
                            true_positives += 1
                        else:
                            false_positives += 1
            
    return true_positives, false_positives, total_objects

def per_image_stats(preds, truth, img_dir, best_cutoff=0.5):
    preds_binary = torch.Tensor(preds>best_cutoff).long()
    tp, fp, fn, tn = smp.metrics.get_stats(
        preds_binary,
        truth,
        mode="binary")
    ids = sorted(os.listdir(img_dir))
    out_df = pd.DataFrame(
        {'basename': [n[:-4] for n in ids],
         'tp': tp.sum(axis=1),
         'fp': fp.sum(axis=1),
         'fn': fn.sum(axis=1),
         'tn': tn.sum(axis=1)}
    )
    return out_df
    

def object_stats(truth_masks, pred_masks, pred_thresh=0.5):
    total_truth_objects = 0
    total_pred_objects = 0
    total_truth_area = 0
    total_pred_area = 0
    max_pred_size = 0
    max_truth_size = 0
    min_truth_size = 1000

    for i in range(len(truth_masks)):
        labeled_truth, max_truth = get_objects(truth_masks[i])
        labeled_pred, max_pred = get_objects(pred_masks[i], pred_thresh)

        if max_truth > 0:
            # First, filter only to truth objects within threshold
            for j in range(1, max_truth+1):
                total_truth_objects += 1
                mask_truth = (labeled_truth==j)
                truth_size = mask_truth.sum()
                total_truth_area += truth_size
                if truth_size > max_truth_size:
                    max_truth_size = truth_size
                if truth_size < min_truth_size:
                    min_truth_size = truth_size
        if max_pred > 0:
            for k in range(1, max_pred+1):
                total_pred_objects += 1
                mask_pred = (labeled_pred==k)
                pred_size = mask_pred.sum()
                total_pred_area += pred_size
                if pred_size > max_pred_size:
                    max_pred_size = pred_size
                if mask_pred.sum() > max_pred_size:
                    max_pred_size = mask_pred.sum()
    out_dict = {
        'total_truth_objects': total_truth_objects,
        'total_truth_area': total_truth_area,
        'min_truth_size': min_truth_size,
        'max_truth_size': max_truth_size,
        'total_pred_objects': total_pred_objects,
        'total_pred_area': total_pred_area,
        'max_pred_size': max_pred_size,
    }
    return out_dict

def size_stats(truth_masks, pred_masks, size_min, size_max, pred_thresh=0.5):
    total_truth_objects = 0
    truth_objects_detected = 0
    total_pred_objects = 0
    pred_objects_false = 0

    for i in range(len(truth_masks)):
        labeled_truth, max_truth = get_objects(truth_masks[i])
        labeled_pred, max_pred = get_objects(pred_masks[i], pred_thresh)

        if max_pred > 0:
            # First, filter only to truth objects within threshold
            for j in range(1, max_truth+1):
                mask_truth = (labeled_truth==j)
                if size_min < mask_truth.sum() <= size_max:
                    total_truth_objects += 1
                    if np.max(labeled_pred * mask_truth) > 0: 
                        truth_objects_detected += 1
            for k in range(1, max_pred+1):
                mask_pred = (labeled_pred==k)
                if size_min < mask_pred.sum() <= size_max:
                    total_pred_objects += 1
                    if np.max(labeled_truth * mask_pred) == 0: 
                        pred_objects_false += 1
    out_dict = {
        'TP': truth_objects_detected,
        'FN': total_truth_objects - truth_objects_detected,
        'TN': total_pred_objects - pred_objects_false,
        'FP': pred_objects_false
    }
    out_dict['recall'] = out_dict['TP'] / (out_dict['TP'] + out_dict['FN'])
    out_dict['precision'] = out_dict['TP'] / (out_dict['TP'] + out_dict['FP'])
    out_dict['f1'] = (out_dict['precision'] + out_dict['recall'])/2
    return out_dict


def pr_curve(preds_path, truth_path):
    preds = np.load(preds_path).flatten()
    truth = np.load(truth_path).flatten()
    return precision_recall_curve(truth, preds)


def full_evaluation(preds_path, truth_path, img_dir, per_image_csv, find_cutoff=True, best_cutoff=0.5, iou_threshold=0.5, crop_to_400=False):
    preds = np.load(preds_path)
    truth = torch.Tensor(np.load(truth_path)).long()
    if crop_to_400: 
        preds = preds[:, 50:450, 50:450]
        truth = truth[:, 50:450, 50:450]

    # Basic pixel-wise stats
    if find_cutoff:
        cutoffs = np.arange(0,1.0,0.01)
        stats_arrays = []
        for cutoff in cutoffs:
            stats_arrays.append(compute_stats(truth, preds, cutoff))
        all_stats = np.vstack(stats_arrays)

        MAX_IOU_MODE = True
        if MAX_IOU_MODE:
            best_index = np.argmax(all_stats[:,1])
        else:
            best_index = np.argmin(np.abs(all_stats[:,2]-all_stats[:,3]))
        best_cutoff = np.median(cutoffs[np.where(all_stats[:,0] == all_stats[best_index, 0])[0]])
    print('Using cutoff:', best_cutoff)
    print('Final pixel-wise stats:', compute_stats(truth, preds, cutoff=best_cutoff))

    print('Object stats', object_stats(truth, preds, pred_thresh=best_cutoff))

    tp, fp, to = calculate_object_p(truth, preds, pred_thresh=best_cutoff, iou_thresh=iou_threshold)
    print('Object Precision:', tp/(tp + fp))
    print('Object Recall:', tp/(to))
    size_dict = {
        'remove': [0, 4],
        'very_small': [4, 10],
        'small': [10, 100],
        'medium': [100, 1000],
        'large': [1000, 5000],
        'xlarge': [5000, 1000000],
    }

    for size_class in ['remove', 'very_small', 'small', 'medium','large', 'xlarge']:
        print(size_stats(truth, preds, size_min=size_dict[size_class][0], size_max=size_dict[size_class][1], pred_thresh=best_cutoff))
    
    per_image_stats(preds, truth, img_dir, best_cutoff=best_cutoff).to_csv(per_image_csv, index=False)


In [ ]:
full_evaluation('./data/preds/reservoirs_10band_manet_datav12_modelv6_val.npy', 
                './data/preds/reservoirs_10band_masks_val.npy',
                './data/reservoirs_10band_v12/img_dir/val',
                './data/preds/val_per_image.csv',
                find_cutoff=False, best_cutoff=0.62, iou_threshold=0.0, crop_to_400=False)

In [ ]:
full_evaluation('./data/preds/reservoirs_10band_manet_datav12_modelv6_test.npy',
                './data/preds/reservoirs_10band_masks_test.npy',
                './data/reservoirs_10band_v12/img_dir/test',
                './data/preds/test_per_image.csv',
                 find_cutoff=False, best_cutoff=0.62, iou_threshold=0.0, crop_to_400=False)

In [ ]:

full_evaluation('./data/preds/reservoirs_10band_manet_datav12_modelv6_train.npy',
                './data/preds/reservoirs_10band_masks_train.npy',
                './data/reservoirs_10band_v12/img_dir/train',
                './data/preds/train_per_image.csv',
                 find_cutoff=False, best_cutoff=0.62, iou_threshold=0.0)

# Per image data - regional stats

In [ ]:
def get_stats(preds, true, cutoff):
    preds_binary = torch.Tensor(preds>cutoff).long()
    tp, fp, fn, tn = smp.metrics.get_stats(
        preds_binary,
        true,
        mode="binary")
    return tp.sum(axis=1), fp.sum(axis=1), fn.sum(axis=1), tn.sum(axis=1)

def per_image_stats(preds_path, truth_path, img_dir, out_path, best_cutoff=0.5):
    preds = np.load(preds_path)
    truth = torch.Tensor(np.load(truth_path)).long()
 
    ids = sorted(os.listdir(img_dir))
    tp, fp, fn, tn = get_stats(preds, truth, cutoff=best_cutoff)
    out_df = pd.DataFrame(
        {'basename': [n[:-4] for n in ids],
         'tp': tp,
         'fp': fp,
         'fn': fn,
         'tn': tn}
    )
    out_df.to_csv(out_path, index=False)
    

In [ ]:
# Training
per_image_stats('./data/preds/reservoirs_10band_manet_datav12_modelv6_train.npy', 
                './data/preds/reservoirs_10band_masks_train.npy',
                './data/reservoirs_10band_v12/img_dir/train/',
                './data/preds/train_per_image.csv',
                best_cutoff=0.62)
# Validation
per_image_stats('./data/preds/reservoirs_10band_manet_datav12_modelv6_val.npy', 
                './data/preds/reservoirs_10band_masks_val.npy',
                './data/reservoirs_10band_v12/img_dir/val/',
                './data/preds/val_per_image.csv',
                best_cutoff=0.62)
# Test
per_image_stats('./data/preds/reservoirs_10band_manet_datav12_modelv6_test.npy', 
                './data/preds/reservoirs_10band_masks_test.npy',
                './data/reservoirs_10band_v12/img_dir/test/',
                './data/preds/test_per_image.csv',
                best_cutoff=0.62)


## Curves: Precision-recall and Loss progression

In [ ]:
training_stats = pd.read_csv('./data/train_csvs/manet_resnet_datav12_modelv6_metrics_clean.csv')
per_epoch = training_stats.groupby('epoch').mean()
to_plot = per_epoch[['train_loss', 'train_dataset_iou', 'valid_loss','valid_dataset_iou']].rename(
    columns={
        'train_loss': 'Training Loss',
        'train_dataset_iou': 'Training IoU',
        'valid_loss':'Validation Loss',
        'valid_dataset_iou':'Validation IoU'
    }
)

In [ ]:
fig, ax = plt.subplots()

colors = ['tab:blue', 'tab:blue', 'tab:red', 'tab:red']
linestyles = ['-', '--', '-', '--']

for col, color, ls in zip(to_plot.columns, colors, linestyles):
    ax.plot(to_plot.index, to_plot[col], color=color, linestyle=ls, label=col, lw=1.2)
ax.legend()
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')

In [ ]:
prec_recall_val = pr_curve('./data/preds/reservoirs_10band_manet_datav12_modelv6_val.npy',
                './data/preds/reservoirs_10band_masks_val.npy')

In [ ]:

prec_recall_test = pr_curve('./data/preds/reservoirs_10band_manet_datav12_modelv6_test.npy',
                './data/preds/reservoirs_10band_masks_test.npy')

In [ ]:
# Combined plot
fig, axs = plt.subplots(1,2, figsize=(7.35,3), constrained_layout=True)

colors = ['tab:blue', 'tab:blue', 'tab:red', 'tab:red']
linestyles = ['-', '--', '-', '--']

for col, color, ls in zip(to_plot.columns, colors, linestyles):
    axs[0].plot(to_plot.index, to_plot[col], color=color, linestyle=ls, label=col, lw=1.2)
axs[0].annotate('A',
                 xy=(1, 1), xycoords='axes fraction',
                 xytext=(-1.3, -1.3), textcoords='offset fontsize',
                 fontsize=12, verticalalignment='bottom', fontfamily='serif')
axs[0].legend(loc='upper right', bbox_to_anchor=(0.94, 1.01))
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Score')
axs[0].set_ylim(0, 1.05)
axs[1].plot(prec_recall_val[0], prec_recall_val[1], label='Validation', color='tab:red')
axs[1].plot(prec_recall_test[0], prec_recall_test[1], label='Test', color='tab:orange')
axs[1].annotate('B',
                 xy=(1, 1), xycoords='axes fraction',
                 xytext=(-1.3, -1.3), textcoords='offset fontsize',
                 fontsize=12, verticalalignment='bottom', fontfamily='serif')
axs[1].legend()
axs[1].set_xlabel('Recall')
axs[1].set_ylabel('Precision')
plt.savefig('/home/ksolvik/research/reservoirs/figs/ch0/training_progress_curve.svg', dpi=300)
plt.savefig('/home/ksolvik/research/reservoirs/figs/ch0/training_progress_curve.jpg', dpi=300,
            pil_kwargs={'quality':95},
            bbox_inches='tight')
